In [1]:
import pandas as pd
import geopandas as gpd
import numpy as np
import os


In [13]:
park_poly_shp = r'C:\laura\NYUSH\Course\8_2026Spring\urban\final_project\公园数据\F\19\上海市_公园面\上海市_公园面.shp'

proj_root = r'C:\laura\NYUSH\Course\8_2026Spring\urban\park_selection'
output_layers_dir = os.path.join(proj_root, 'dataset_structured/05_processed/standardized_layers')
cleaned_poi_csv = os.path.join(proj_root, 'dataset_structured/05_processed/cleaned_target_pois.csv')

if not os.path.exists(output_layers_dir):
    os.makedirs(output_layers_dir)


In [3]:
# define weights for Qj calculation (can be adjusted based on importance)
weights = {
    'w1_area': 0.5,
    'w2_bus': 0.2,
    'w3_toilet': 0.2,
    'w4_pharmacy': 0.1
}

# 中心城区过滤名单 (保持一致性)
central_districts = ['黄浦区', '徐汇区', '长宁区', '静安区', '普陀区', '虹口区', '杨浦区']


In [9]:
print("Loading and Standardizing Polygons...")
gdf_parks = gpd.read_file(park_poly_shp)

gdf_parks = gdf_parks.to_crs("EPSG:3857")

gdf_parks['real_area_m2'] = gdf_parks.geometry.area

Loading and Standardizing Polygons...


In [10]:
# --- 任务 B: 区域过滤 (只做中心城区) ---
# 检查公园 Shapefile 是否有行政区字段，如果有，进行过滤
# 假设有字段 'district'，如果没有，请根据实际字段名修改或跳过
if 'district' in gdf_parks.columns:
    print(f"Filtering Parks in Central Districts...")
    gdf_parks = gdf_parks[gdf_parks['district'].isin(central_districts)].copy()

In [14]:

# --- 任务 C: 加载 POI ---
print("Loading Amenity POIs...")
df_poi = pd.read_csv(cleaned_poi_csv)
# 将 POI 表格激活为 3857 点
gdf_poi_3857 = gpd.GeoDataFrame(
    df_poi,
    geometry=gpd.points_from_xy(df_poi.lon_wgs, df_poi.lat_wgs),
    crs="EPSG:4326"
).to_crs("EPSG:3857")


Loading Amenity POIs...


In [15]:

# ==========================================
# 3. 计算公园服务半径内的设施配套 (Qj)
# ==========================================
# 严格按照 Proposal 的 500m 缓冲区 (15-min walk)
print("Calculating Single Park Quality Index (Qj)...")
parks_buffer = gdf_parks.copy()
parks_buffer['geometry'] = parks_buffer.geometry.buffer(500)

# 空间连接：看哪个 POI 在哪个公园缓冲区的 500m 圈内
# Predicate='within'保证点在面内
joined = gpd.sjoin(gdf_poi_3857, parks_buffer, how='inner', predicate='within')

# 按公园 ID 统计配套数量
# index_right 是 joined 表格中 parks_buffer 的原始索引
amenity_counts = joined.groupby(['index_right', 'category_group']).size().unstack(fill_value=0)



Calculating Single Park Quality Index (Qj)...


In [16]:
# ==========================================
# 4. caluclate Qj
# ==========================================

gdf_parks_final = gdf_parks.join(amenity_counts, how='left').fillna(0)

gdf_parks_final['Qj_index'] = (
    np.log1p(gdf_parks_final['real_area_m2']) * weights['w1_area'] + 
    gdf_parks_final.get('Transport', 0) * weights['w2_bus'] + 
    gdf_parks_final.get('Toilet', 0) * weights['w3_toilet'] + 
    gdf_parks_final.get('Medical_Service', 0) * weights['w4_pharmacy']
)

# normalize Qj to [0,1] for easier comparison
max_q = gdf_parks_final['Qj_index'].max()
if max_q > 0:
    gdf_parks_final['norm_Qj'] = gdf_parks_final['Qj_index'] / max_q
else:
    gdf_parks_final['norm_Qj'] = 0



In [17]:
# save results
save_path = os.path.join(output_layers_dir, 'parks_with_quality_3857.shp')
gdf_parks_final.to_file(save_path, encoding='utf-8')

print(f"\n--- SUCCESS: Qj calculated for {len(gdf_parks_final)} individual parks. ---")
print(f"Result saved as: {save_path}")


--- SUCCESS: Qj calculated for 1733 individual parks. ---
Result saved as: C:\laura\NYUSH\Course\8_2026Spring\urban\park_selection\dataset_structured/05_processed/standardized_layers\parks_with_quality_3857.shp


C:\Users\laura\AppData\Local\Temp\ipykernel_32656\293866273.py:3: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdf_parks_final.to_file(save_path, encoding='utf-8')
c:\Users\laura\anaconda3\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'real_area_m2' to 'real_area_'
  ogr_write(
c:\Users\laura\anaconda3\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'Medical_Service' to 'Medical_Se'
  ogr_write(
